In [ ]:
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
from scipy.integrate import odeint
import matplotlib.pyplot as plt

In [ ]:
# Parameters
L = 0.25  #Độ dài dây treo con lắc
g = 9.81  #Gia tốc trọng trường
w = np.sqrt(g/L) #Tần số góc của con lắc
k = w**2 #Hệ số trong phương trình vi phân
end_time = 15 #Thời gian mô phỏng

In [ ]:
# ODE function for a simple pendulum exact solution
def simple_pendulum_eqn(state, t, L, g):
    theta, theta_dot = state
    theta_ddot = -k*np.sin(theta) #tính gia tốc góc
    return [theta_dot, theta_ddot] #trả về hệ phương trình bậc nhất

# Initial state [theta, theta_dot]
initial_state = [np.pi/4, 0] #điều kiện ban đầu, vị trí ban đầu và vận tốc ban đầu

# Small Angle Approximation Analytical Solution
def pendulum_solution(w, x):
    # Ensure x is a numpy array for numpy.cos operation to avoid DeprecationWarning
    x_val = x.detach().cpu().numpy() if isinstance(x, torch.Tensor) else x #chuyển tensor sang numpy
    return initial_state[0]*np.cos(w*x_val) #tính nghiệm

In [ ]:
# Create a one-dimensional time array
t = np.linspace(0, end_time, 50000)
x = torch.tensor(t, dtype=torch.float32).view(-1, 1)

# Numerical solution of the simple pendulum ODEs
states = odeint(simple_pendulum_eqn, initial_state, t, args=(L, g))
y = torch.tensor(states[:, 0], dtype=torch.float32).view(-1, 1)
print(x.shape, y.shape)

x_data = x[:10000:1000]
y_data = y[:10000:1000]

print(x_data.shape, y_data.shape)

In [ ]:
# View the analytical solution
View_Analytical_Solution = True

if View_Analytical_Solution:
    fig = plt.figure(figsize=(8, 5))

    # ===== nghiệm chính xác =====
    plt.plot(
        x,
        y,
        linewidth=2,
        label='Nghiệm chính xác'
    )

    # ===== nghiệm xấp xỉ góc nhỏ =====
    plt.plot(
        x,
        pendulum_solution(w, x),
        '--',
        linewidth=2,
        label='Xấp xỉ góc nhỏ'
    )

    # ===== tiêu đề =====
    plt.title(
        'So sánh nghiệm chính xác và nghiệm xấp xỉ góc nhỏ',
        fontsize=14
    )

    # ===== nhãn trục =====
    plt.xlabel('Thời gian (s)', fontsize=12)
    plt.ylabel(r'Góc lệch $\theta(t)$ (rad)', fontsize=12)

    # ===== lưới =====
    plt.grid(True, alpha=0.5)

    # ===== chú thích =====
    plt.legend(fontsize=11)

    # ===== cỡ chữ trục =====
    plt.xticks(fontsize=10)
    plt.yticks(fontsize=10)

    # ===== ghi điều kiện đầu phía dưới hình =====
    fig.text(
        0.5,
        0.01,
        r'Điều kiện đầu: $\theta(0)=\frac{\pi}{4},\ \omega(0)=0$',
        ha='center',
        fontsize=11
    )

    # ===== căn chỉnh =====
    plt.tight_layout(rect=[0, 0.05, 1, 1])

    # ===== lưu file =====
    plt.savefig(
        'solution.pdf',
        bbox_inches='tight'
    )

    plt.show()

In [ ]:
#Neural network
class FCN(nn.Module):
    "Defines a connected network"

    def __init__(self, N_INPUT, N_OUTPUT, N_HIDDEN, N_LAYERS):
        super().__init__()
        activation = nn.Tanh
        self.fcs = nn.Sequential(*[
                        nn.Linear(N_INPUT, N_HIDDEN),
                        activation()])
        self.fch = nn.Sequential(*[
                        nn.Sequential(*[
                            nn.Linear(N_HIDDEN, N_HIDDEN),
                            activation()]) for _ in range(N_LAYERS-1)])
        self.fce = nn.Linear(N_HIDDEN, N_OUTPUT)

    def forward(self, x):
        x = torch.cat([x, torch.sin(x), torch.cos(x)], dim=1)
        x = self.fcs(x)
        x = self.fch(x)
        x = self.fce(x)
        return x

In [ ]:
def save_gif_PIL(outfile, files, fps=5, loop=0):
    "Helper function for saving GIFs"
    imgs = [Image.open(file) for file in files]
    imgs[0].save(fp=outfile, format='GIF', append_images=imgs[1:], save_all=True, duration=int(1000/fps), loop=loop)

def plot_result(x, y, x_data, y_data, yh, step, xp=None):
    "Pretty plot training results"
    plt.figure(figsize=(8,4))
    plt.plot(x[::150], y[::150], marker='o', linestyle='None',
         color="tab:red", alpha=0.5, label="Numerical solution")
    plt.plot(x,yh, color="tab:blue", linewidth= 3, alpha=1, label="NN prediction")
    plt.scatter(x_data, y_data, s=60, color="green", alpha=0.8, label='Training data', marker='s')
    plt.axvline(x= x_split, color='black', linestyle='--', linewidth=1)
    l = plt.legend(loc=(1.01,0.34), frameon=False, fontsize="large")
    plt.setp(l.get_texts(), color="k")
    plt.xlim(-0.05, end_time+0.05)
    plt.ylim(-1.1, 1.1)
    # plt.text(1.065,0.95,f"Training step: {step}",fontsize="xx-large",color="k") # Removed Training step text
    plt.grid(True)
    # plt.show() # Removed plt.show() to allow further modification

In [ ]:
# train standard neural network to fit training data
torch.manual_seed(123)
model = FCN(3,1,32,3)
optimizer = torch.optim.Adam(model.parameters(),lr=1e-3)

files = []
x_split = 3 # Define x_split here
for i in range(5000):
    optimizer.zero_grad()
    yh = model(x_data)
    loss = torch.mean((yh-y_data)**2)# use mean squared error
    loss.backward()
    optimizer.step()

    # plot the result as training progresses
    if (i+1) % 10 == 0:

        yh = model(x).detach()

        plot_result(x, y, x_data, y_data, yh, i+1) # Pass i+1 as step

        file = "nn_%.8i.png"%(i+1)
        plt.savefig(file, bbox_inches='tight', pad_inches=0.1, dpi=100, facecolor="white")
        files.append(file)

        if (i+1) % 500 == 0: plt.show()
        else: plt.close("all")

save_gif_PIL("nn.gif", files, fps=20, loop=0)

In [ ]:
x_physics = torch.linspace(0, end_time, 500).view(-1,1).requires_grad_(True)# sample locations over the problem domain
lambda1 = 0.05
lambda2 = 100
lambda_ic = 750
lambda_energy = 10

model = FCN(3,1,128,6)
optimizer = torch.optim.Adam(model.parameters(),lr=5e-4)
files = []

# Initialize lists to store loss values
loss1_history = []
loss2_history = []
loss_ic_history = []
loss_energy_history = []
total_loss_history = []
training_steps = []

theta0 = torch.tensor([[np.pi/4]])
omega0 = torch.tensor([[0.0]])

for i in range(30000):
    optimizer.zero_grad()

    # compute the "data loss"
    yh = model(x_data)
    loss1 = torch.mean((yh-y_data)**2)# use mean squared error

    # compute the "physics loss"
    yhp = model(x_physics)
    dx  = torch.autograd.grad(yhp, x_physics, torch.ones_like(yhp), create_graph=True)[0]# computes dy/dx
    dx2 = torch.autograd.grad(dx,  x_physics, torch.ones_like(dx),  create_graph=True)[0]# computes d^2y/dx^2
    physics = dx2 + k*torch.sin(yhp)
    loss2 = torch.mean(physics**2)

    # initial condition
    x0 = torch.tensor([[0.0]], requires_grad=True)
    y0 = model(x0)
    dy0 = torch.autograd.grad(
        y0, x0,
        torch.ones_like(y0),
        create_graph=True
    )[0]
    loss_ic = (y0 - theta0)**2 + (dy0 - omega0)**2

    #energy
    dy = dx  # theta'
    E = 0.5 * dy**2 + k * (1 - torch.cos(yhp))
    E0 = E[0].detach()  # năng lượng ban đầu
    loss_energy = torch.mean((E - E0)**2)

    # backpropagate joint loss
    loss = lambda1*loss1 + lambda2*loss2 + lambda_ic*loss_ic + lambda_energy*loss_energy
    loss.backward()
    optimizer.step()

    # Store loss values
    loss1_history.append(loss1.item())
    loss2_history.append(loss2.item())
    loss_ic_history.append(loss_ic.item())
    loss_energy_history.append(loss_energy.item())
    total_loss_history.append(loss.item())
    training_steps.append(i + 1)

    # Print loss values every 1000 steps
    if (i + 1) % 1000 == 0:
        print(f"Step {i+1}, Loss1: {loss1.item():.6f}, Loss2: {loss2.item():.6f}, Loss_ic: {loss_ic.item():.6f}, Loss_energy: {loss_energy.item():.6f}, Total Loss: {loss.item():.6f}")

    # plot the result as training progresses
    if (i+1) % 150 == 0:

        yh = model(x).detach()
        xp = x_physics.detach()

        plot_result(x,y,x_data,y_data,yh,i+1,xp)

        file = "pinn_%.8i.png"%(i+1)
        plt.savefig(file, bbox_inches='tight', pad_inches=0.1, dpi=100, facecolor="white")
        files.append(file)

        if (i+1) % 6000 == 0: plt.show()
        else: plt.close("all")

save_gif_PIL("pinn.gif", files, fps=20, loop=0)


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# training steps
training_steps = np.arange(len(total_loss_history))

plt.figure(figsize=(12, 7))

# từng thành phần loss
plt.plot(
    training_steps,
    loss1_history, # Changed from loss_data_history
    label='Data Loss',
    linewidth=2
)

plt.plot(
    training_steps,
    loss2_history, # Changed from loss_physics_history
    label='Physics Loss',
    linewidth=2
)

plt.plot(
    training_steps,
    loss_ic_history,
    label='Initial Condition Loss',
    linewidth=2
)

plt.plot(
    training_steps,
    loss_energy_history,
    label='Energy Loss',
    linewidth=2
)

# loss tổng
plt.plot(
    training_steps,
    total_loss_history,
    '--',
    label='Total Loss',
    linewidth=2
)

plt.xlabel('Training Steps', fontsize=14)
plt.ylabel('Loss Value', fontsize=14)

# log scale rất quan trọng
plt.yscale('log')

plt.legend(fontsize=11)
plt.grid(True, which="both", ls="-", alpha=0.6)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error
import numpy as np

# Generate predictions for the entire time range using the current trained model
x_feature = torch.cat([
    x,
    torch.sin(20*x),
    torch.cos(20*x)
], dim=1)

yh = model(x).detach()

# Calculate MSE
mse = mean_squared_error(y.detach().numpy(), yh.numpy())
print(f"Mean Squared Error (MSE): {mse}")

# Calculate RMSE
rmse = np.sqrt(mse)
print(f"Root Mean Squared Error (RMSE): {rmse}")

# Calculate MAE
mae = mean_absolute_error(y.detach().numpy(), yh.numpy())
print(f"Mean Absolute Error (MAE): {mae}")

#R^2
import torch
import numpy as np

def r2_score_pinn(theta_true, theta_pred):
    theta_true = np.array(theta_true)
    theta_pred = np.array(theta_pred)

    ss_res = np.sum((theta_true - theta_pred) ** 2)
    ss_tot = np.sum((theta_true - np.mean(theta_true)) ** 2)

    return 1 - ss_res / ss_tot

# Assign theta_true and theta_pred before calling the function
theta_true = y.numpy()
theta_pred = yh.numpy()

# Calculate R^2 using the globally available theta_true and theta_pred
r2 = r2_score_pinn(theta_true, theta_pred)
print(f"R^2 = {r2:.6f}")